# Multi-Position Execution Backtest

This notebook shows the mechanics behind a multi-position prediction-market backtest using synthetic order-book data.

The point is not to present a tradeable result. The point is to make the state management visible: several positions can be open at once, each position exits on its own outcome side, and execution depends on displayed liquidity rather than a single top-of-book price.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from backtesting.multi_position_backtester import (
    estimate_buy_fill,
    estimate_sell_fill,
    max_allowed_spread,
    run_multi_position_backtest,
    summarise_trade_ledger,
)


## Synthetic Order Book

I create a small toy market with Yes and No rows at each timestamp. The columns mirror the fields needed for execution accounting: bid/ask prices, displayed sizes, a rolling signal, staleness, and a desired trade size.


In [ ]:
rng = np.random.default_rng(7)
timestamps = pd.date_range("2026-01-05 14:30:00+00:00", periods=120, freq="1min")

rows = []
for minute, timestamp in enumerate(timestamps):
    signal = np.sin(minute / 11) * 2.8 + rng.normal(0, 0.25)
    for outcome in ["Yes", "No"]:
        mid = 0.50 + 0.0015 * minute if outcome == "Yes" else 0.50 - 0.0015 * minute
        spread = 0.02 + 0.01 * (minute % 17 == 0)
        rows.append(
            {
                "timestamp_utc": timestamp,
                "outcome": outcome,
                "bid_price_1": max(mid - spread / 2, 0.01),
                "bid_size_1": 8 + rng.integers(0, 8),
                "bid_price_2": max(mid - spread / 2 - 0.015, 0.01),
                "bid_size_2": 12 + rng.integers(0, 10),
                "bid_price_3": max(mid - spread / 2 - 0.035, 0.01),
                "bid_size_3": 20 + rng.integers(0, 12),
                "ask_price_1": min(mid + spread / 2, 0.99),
                "ask_size_1": 8 + rng.integers(0, 8),
                "ask_price_2": min(mid + spread / 2 + 0.015, 0.99),
                "ask_size_2": 12 + rng.integers(0, 10),
                "ask_price_3": min(mid + spread / 2 + 0.035, 0.99),
                "ask_size_3": 20 + rng.integers(0, 12),
                "rolling_deviation": signal,
                "poly_stale_mins": minute % 6,
                "trade_size": 6.0,
            }
        )

trade_table = pd.DataFrame(rows)
trade_table["poly_spread"] = trade_table["ask_price_1"] - trade_table["bid_price_1"]
trade_table["spread_over_ask"] = trade_table["poly_spread"] / trade_table["ask_price_1"]
trade_table["liquid_book"] = (
    (trade_table["poly_spread"] <= 0.08)
    & (trade_table["spread_over_ask"] <= 0.20)
    & (trade_table["bid_price_1"] >= 0.04)
)

trade_table.head()


## Entry Signals

I keep the example signal deliberately simple. A large positive deviation buys Yes; a large negative deviation buys No. The signal is kept separate from execution. A row can have a signal and still fail to trade if the book is too wide or too shallow.


In [ ]:
Z_ENTRY = 1.96

trade_table["entry_side"] = None
trade_table.loc[
    (trade_table["rolling_deviation"] >= Z_ENTRY) & (trade_table["outcome"] == "Yes"),
    "entry_side",
] = "Yes"
trade_table.loc[
    (trade_table["rolling_deviation"] <= -Z_ENTRY) & (trade_table["outcome"] == "No"),
    "entry_side",
] = "No"

trade_table[trade_table["entry_side"].notna()].head(10)


## Row-Level Spread Tolerance

The allowed spread is not fixed. It starts from a conservative base, widens for stronger signals and healthier displayed depth, and stays below a hard cap.


In [ ]:
example_signal_row = trade_table[trade_table["entry_side"].notna()].iloc[0]

allowed_spread = max_allowed_spread(example_signal_row)
buy_fill = estimate_buy_fill(
    example_signal_row,
    example_signal_row["trade_size"],
    max_spread=allowed_spread,
)

allowed_spread, buy_fill


## Multi-Position Backtest

The backtester can hold multiple positions at the same time. It checks existing positions first, carries forward survivors, opens new positions when execution passes, and closes remaining positions on the last available row for their own outcome side.


In [ ]:
trades, debug = run_multi_position_backtest(
    trade_table,
    max_hold_minutes=60,
    take_profit=0.12,
    stop_loss=0.05,
    stop_loss_activation_minutes=10,
)

summary = pd.DataFrame([summarise_trade_ledger(trades)])

display(summary)
display(trades.head())
display(debug.head())


## Sanity Checks

I check entries, exits, and open-position counts before interpreting any P&L. This is the part that usually catches mistakes in state handling.


In [ ]:
sanity = {
    "entry_rows": int(debug["entry_taken"].sum()),
    "closed_trades": int(len(trades)),
    "max_open_positions": int(debug["open_positions"].max()) if len(debug) else 0,
    "forced_closes": int((trades["exit_reason"] == "end_of_window").sum()) if len(trades) else 0,
}

pd.DataFrame([sanity])


## Interpretation

This synthetic notebook demonstrates mechanics only. It does not use real market data, production market identifiers, private APIs, or fitted strategy parameters.

The useful part is the structure: signal generation is separate from execution, order-book levels determine fills, and positions are closed on the correct outcome side.
